##DLT Pipelines

In [0]:
df = spark.read.format('delta').load('abfss://silver@datalakevishnu1.dfs.core.windows.net/products')

In [0]:
df.display()

In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

#Streaming Table

##Expectations

In [0]:
my_rules =  {
    "rule1":"product_id is NOT NULL",
    "rule2":"product_name is NOT NULL",
}

In [0]:
@dlt.table()

@dlt.expect_all_or_drop(my_rules)
def DimProducts_stg():
    df = spark.readStream.table('databricksreddy.silver.products')
    return df



###Streaming Views

In [0]:
@dlt.view
def DimProducts_View():
    df = spark.readStream.table("Live.DimProducts_stg")
    return df

##DimProducts

In [0]:
dlt.create_streaming_table("DimProductsNew")

In [0]:
dlt.apply_changes(
    target = "DimProductsNew",
    source = "Live.DimProducts_View",
    keys = ["product_id"],
    sequence_by = "product_id",
    stored_as_scd_type="2"
)